# Threads Bot — инструментарий для JSON-аккаунтов (Cloudflare)

Ноутбук поячеечно. Каждая ячейка самодостаточна, но опирается на переменные
предыдущих. Работает и в Google Colab, и в Jupyter.

Что делает:
1. Загрузка JSON-файлов cookies (Cookie-Editor / Playwright).
2. Проверка каждого файла: валидность, ключевые cookies, срок жизни.
3. Нормализация в единый формат (`sameSite`, приведение `expires`).
4. Генерация одного SQL-скрипта для импорта в Cloudflare D1.
5. Экспорт нормализованных JSON обратно в ZIP (для переноса).
6. Очистка временных файлов.


## Ячейка 1 — загрузка JSON-файлов

В Colab откроется диалог выбора. Можно выбрать несколько файлов сразу.
В Jupyter — положи `.json` рядом с ноутбуком и просто пропусти эту ячейку.


In [ ]:
import os, json, time, io

UPLOADED = {}
try:
    from google.colab import files
    UPLOADED = files.upload()  # dict: filename -> bytes
    print(f'Загружено через Colab: {len(UPLOADED)} файлов')
except Exception:
    # Jupyter / локальный запуск: подберём все *.json из текущей папки
    for fname in sorted(os.listdir('.')):
        if fname.endswith('.json'):
            with open(fname, 'rb') as fh:
                UPLOADED[fname] = fh.read()
    print(f'Найдено локально: {len(UPLOADED)} файлов')

for fname in UPLOADED:
    print('  •', fname)


## Ячейка 2 — проверка + нормализация

Для каждого файла:
- проверяем, что это массив объектов с `name`/`value`;
- ищем ключевые cookies `sessionid`/`ds_user_id`/`ig_did`;
- находим самую раннюю дату истечения;
- нормализуем `sameSite`, приводим `expirationDate`→`expires`;
- складываем в `NORMALIZED[name]` для следующих ячеек.


In [ ]:
KEY = {'sessionid', 'ds_user_id', 'ig_did', 'session_id'}
WARN_DAYS = 7
NORMALIZED = {}  # name -> list[cookie]
REPORT = []      # человекочитаемый отчёт

def normalize_same_site(v):
    v = str(v or 'Lax')
    low = v.lower()
    if low in ('unspecified', 'null', ''): return 'Lax'
    if low in ('no_restriction', 'none'):  return 'None'
    return v[0].upper() + v[1:].lower()

def normalize_one(cookies):
    out = []
    for c in cookies:
        if not isinstance(c, dict) or 'name' not in c or 'value' not in c: 
            raise ValueError('элемент без name/value')
        item = {
            'name':     c['name'],
            'value':    c['value'],
            'domain':   c.get('domain', '.threads.com'),
            'path':     c.get('path', '/'),
            'httpOnly': bool(c.get('httpOnly', False)),
            'secure':   c.get('secure', True) is not False,
            'sameSite': normalize_same_site(c.get('sameSite')),
        }
        exp = c.get('expirationDate') or c.get('expires')
        if isinstance(exp, (int, float)) and exp > 0:
            item['expires'] = int(exp)
        out.append(item)
    return out

for fname, blob in UPLOADED.items():
    name = os.path.splitext(fname)[0]
    line = f'📄 {fname} → {name}: '
    try:
        data = json.loads(blob.decode('utf-8'))
        assert isinstance(data, list) and data, 'JSON не массив или пуст'
        norm = normalize_one(data)
        names = {c['name'] for c in norm}
        missing = KEY - names
        exps = [c['expires'] for c in norm if 'expires' in c]
        earliest = min(exps) if exps else None
        srok = time.strftime('%d.%m.%Y', time.gmtime(earliest)) if earliest else 'без срока'
        marks = []
        if not (names & KEY):
            marks.append('❌ нет ключевых cookies (sessionid/ds_user_id/ig_did)')
        elif missing:
            marks.append(f'⚠️ отсутствуют: {",".join(sorted(missing))}')
        if earliest is not None:
            days = (earliest - time.time()) / 86400
            if days < 0: marks.append(f'❌ cookies истекли ({srok})')
            elif days < WARN_DAYS: marks.append(f'⚠️ скоро истекают ({srok}, {int(days)}д)')
            else: marks.append(f'✅ до {srok}')
        NORMALIZED[name] = norm
        line += ('; '.join(marks) if marks else 'ok') + f' | {len(norm)} cookies'
    except Exception as e:
        line += f'❌ ошибка: {e}'
    REPORT.append(line)
    print(line)


## Ячейка 3 — сгенерировать `import.sql` для Cloudflare D1

На выходе — файл `import.sql`, который можно применить одной командой:
```bash
wrangler d1 execute threadsbot --remote --file=./import.sql
```
или (локально):
```bash
wrangler d1 execute threadsbot --local --file=./import.sql
```

SQL использует `INSERT ... ON CONFLICT(name) DO UPDATE` — можно запускать повторно, безопасно.


In [ ]:
if not NORMALIZED:
    raise RuntimeError('Сначала прогони ячейки 1 и 2')

ts = time.strftime('%Y-%m-%dT%H:%M:%S.000Z', time.gmtime())
def q(v):  # эскейп строки для SQL
    return "'" + v.replace("'", "''") + "'"

lines = [
    '-- Cloudflare D1 import для threads_accounts',
    f'-- Сгенерировано: {ts}',
    f'-- Аккаунтов: {len(NORMALIZED)}',
    'BEGIN TRANSACTION;',
]
for name, cookies in NORMALIZED.items():
    raw = json.dumps(cookies, ensure_ascii=False)
    lines.append(
        'INSERT INTO threads_accounts(name,cookies,enabled,is_alive,hourly_reset,updated_at) '
        f'VALUES({q(name)},{q(raw)},1,1,{q(ts)},{q(ts)}) '
        'ON CONFLICT(name) DO UPDATE SET '
        'cookies=excluded.cookies,enabled=1,is_alive=1,last_error=NULL,updated_at=excluded.updated_at;'
    )
lines.append('COMMIT;')

with open('import.sql', 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(lines))

print(f'Записано {len(NORMALIZED)} аккаунтов в import.sql ({os.path.getsize("import.sql")} байт)')
try:
    from google.colab import files as _cf
    _cf.download('import.sql')
except Exception:
    print('Скачай import.sql из панели файлов слева')


## Ячейка 3b — опционально: залить SQL сразу в Cloudflare D1

Если нет локального `wrangler`, можно отправить `import.sql` в D1 через API.
Токен вводится через `getpass` и не печатается. Database ID уже стоит из `wrangler.toml`.
Ячейку можно пропустить и залить файл руками:
```bash
wrangler d1 execute threadsbot --remote --file=./import.sql
```


In [ ]:
import json, urllib.request, getpass
from pathlib import Path

if not Path('import.sql').exists():
    raise RuntimeError('Сначала ячейка 3 — нужен import.sql')

CF_ACCOUNT = getpass.getpass('Cloudflare Account ID (скрыто): ').strip()
CF_TOKEN = getpass.getpass('Cloudflare API Token (скрыто): ').strip()
DB_ID = '831162d2-03cc-4525-a4b8-0651e2c20e61'  # wrangler.toml database_id

sql = Path('import.sql').read_text(encoding='utf-8')
statements = []
for chunk in sql.split(';'):
    stmt = chunk.strip()
    if not stmt or stmt.startswith('--'):
        continue
    if stmt in ('BEGIN TRANSACTION', 'COMMIT'):
        continue
    statements.append(stmt)
print(f'Отправляю {len(statements)} statement(s) в D1...')

url = f'https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT}/d1/database/{DB_ID}/query'
ok = 0
for stmt in statements:
    req = urllib.request.Request(
        url,
        data=json.dumps({'sql': stmt}).encode(),
        headers={'Authorization': f'Bearer {CF_TOKEN}', 'Content-Type': 'application/json'},
        method='POST',
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            payload = json.loads(resp.read().decode())
        if payload.get('success'):
            ok += 1
        else:
            print('❌', payload)
    except Exception as e:
        print('❌', e)
print(f'Готово: {ok}/{len(statements)}')
CF_TOKEN = CF_ACCOUNT = ''



## Ячейка 4 — экспорт нормализованных JSON обратно (ZIP)

Если хочешь оставить у себя нормализованные `.json` для переноса в бот руками
(через `/account_export` / загрузку файла), эта ячейка соберёт их в один ZIP.


In [ ]:
import zipfile

if not NORMALIZED:
    raise RuntimeError('Сначала прогони ячейки 1 и 2')

zip_path = 'accounts_normalized.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name, cookies in NORMALIZED.items():
        zf.writestr(f'{name}.json', json.dumps(cookies, ensure_ascii=False, indent=2))

print(f'Готов ZIP: {zip_path} ({os.path.getsize(zip_path)} байт, {len(NORMALIZED)} файлов)')
try:
    from google.colab import files as _cf
    _cf.download(zip_path)
except Exception:
    print('Скачай accounts_normalized.zip из панели файлов')


## Ячейка 5 — очистка временных файлов

Cookies = ключи от аккаунтов. Убираем их из Colab, чтобы ничего не осталось
в скачиваемом состоянии окружения.


In [ ]:
removed = 0
for fname in list(os.listdir('.')):
    if fname.endswith(('.json', '.sql', '.zip')):
        try: os.remove(fname); removed += 1; print('  удалено:', fname)
        except Exception as e: print('  ошибка:', fname, e)
UPLOADED.clear(); NORMALIZED.clear(); REPORT.clear()
print(f'\nВсего удалено: {removed}. Переменные очищены.')


## Памятка по банам

1. **Одна сессия = один IP**. Не запускай тот же JSON в двух местах одновременно.
2. Не превышай **20 запросов/час** на аккаунт (в боте это уже настроено).
3. Cookies обновляются автоматически после каждого успешного запроса.
4. Мёртвые сессии (редирект на `/login`) нужно пере-экспортировать вручную.
5. Свежие аккаунты (< 2 дней активности) банят быстрее — прогони «руками» сначала.
6. В боте: `/accounts`, `/account_check`, `/account_del`, `/account_export`, загрузка `.json` файлом.
7. Каждые 6 часов бот сам диагностирует все JSON и шлёт админам алерт, если что-то не так.
